# OMNI-X14 · Is Cosmos-Embed1 better than InternVideo2 for actions? (Colab GPU)

Runs the repo's own code from a clone of the branch: `scripts/encode_virat_cosmos.py` (the Cosmos process, the
timing, the tuning), `scripts/measure_sm1.py`, `scripts/tune_fusion.py`, fusion and the classifier, and ai_workers'
SigLIP 2 and InternVideo2.

**Two processes, two transformers versions.** Cosmos-Embed1's code needs transformers 4.51.3 (it does not run on
5.x); SigLIP 2's query vectors must be made on the repo's pinned 5.20.1, as SM-1 made them. So Cosmos runs in a
process of its own with 4.51.3 first on its path (step 3), and writes everything it makes to Drive; this notebook's
process keeps 5.20.1 and reads them.

1. **Setup:** clone, install the pinned versions; transformers 4.51.3 into `/content/cosmos_env` for Cosmos only
2. **Models (this process):** SigLIP 2's text side, InternVideo2, the classifier; Cosmos's weights downloaded to Drive
3. **The Cosmos process:** the text vectors of the 100 queries, its GPU time per clip in both spacings, then both
   spacings encoded for the same 275 videos and 35,704 kept frames, never re-gated: **A `1s`** (Algorithm 2's
   window) and **B `4s`** (8 frames over the 4 s before `t`). Resumable: if the session ends, run all again
4. **GPU time:** InternVideo2 on the same 64 clips, beside Cosmos; cameras per GPU
5. **The labels:** OMNI-709's chance level, before any search
6. **Tune** the fusion weights per spacing on the tuning queries only, as D-40 did
7. **SM-1** per spacing; the better spacing; the ticket's rule
8. **Save** to `MyDrive/omnisight/results/sm1_result_cosmos.json`

Runtime → Change runtime type → **GPU** (L4 or A100). Then Run all.

In [ ]:
# 1 · Setup
BRANCH = "slice-09-10-forecast"
REPO = "https://github.com/OmniSight-team/omnisight.git"

import os, subprocess, sys
if not os.path.isdir("/content/omni-sight"):
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO, "/content/omni-sight"], check=True)
else:
    subprocess.run(["git", "-C", "/content/omni-sight", "pull", "-q"], check=True)
COMMIT = subprocess.run(["git", "-C", "/content/omni-sight", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout.strip()
print(COMMIT)

# This process: the versions in uv.lock; Colab keeps its own GPU build of torch and torchvision.
!pip install -q "transformers==5.20.1" "huggingface-hub==2.2.0" "safetensors==0.8.0" "open-clip-torch==3.3.0" -e /content/omni-sight/services/common
# The Cosmos process only: transformers 4.51.3 and what it needs that Colab lacks, in a folder of its own.
# --no-deps, so Colab's numpy, torch and the rest are used as they are.
!pip install -q --no-deps --target /content/cosmos_env "transformers==4.51.3" "tokenizers>=0.21,<0.22" "huggingface-hub>=0.30,<1.0" einops

sys.path.insert(0, "/content/omni-sight/services/common")
sys.path.insert(0, "/content/omni-sight/services/ai_workers")
sys.path.insert(0, "/content/omni-sight/scripts")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import torch
assert torch.cuda.is_available(), "no GPU: Runtime -> Change runtime type -> GPU"
print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# 2 · This process: SigLIP 2's text side, InternVideo2 and the classifier (pinned, as SM-1 ran); Cosmos's weights on Drive
import json
from pathlib import Path
from omnisight_common.fusion import CLASSIFIER_FILE, ModalityClassifier
from omnisight_common.settings import CommonSettings
from app.encoder import FrameTextEncoder, ensure_checkpoint
from app import clip_encoder
from app.clip_encoder import ClipTextEncoder
import encode_virat_archive as archive_runner
import encode_virat_cosmos as cosmos_runner
import measure_sm1 as sm1
import tune_fusion

REPO_DIR = Path("/content/omni-sight")
DRIVE = Path("/content/drive/MyDrive/omnisight")
MODELS = DRIVE / "models" / "pretrained"
VIRAT = DRIVE / "data" / "virat"
VIDEOS = VIRAT / "videos_original"
IV2_ARCHIVE = DRIVE / "embeddings" / "virat"
COSMOS_ARCHIVE = DRIVE / "embeddings" / "virat_cosmos"

iv2_pins = {name: CommonSettings.model_fields[name].default for name in
            ("frame_model_id", "frame_model_revision", "clip_model_id", "clip_model_revision")}
cosmos_pins = {**iv2_pins, "clip_model_id": cosmos_runner.COSMOS_ID, "clip_model_revision": cosmos_runner.COSMOS_REVISION}
print("pins:", cosmos_pins)
siglip_dir = ensure_checkpoint(iv2_pins["frame_model_id"], iv2_pins["frame_model_revision"], MODELS / "siglip2-so400m-patch14-384")
siglip = FrameTextEncoder(siglip_dir, device="cuda")
iv2_dir = ensure_checkpoint(iv2_pins["clip_model_id"], iv2_pins["clip_model_revision"], MODELS / "internvideo2-clip-s", clip_encoder.ALLOW_PATTERNS)
iv2 = ClipTextEncoder(iv2_dir, device="cuda")
classifier = ModalityClassifier.load(iv2_pins["frame_model_id"], iv2_pins["frame_model_revision"])
cosmos_dir = cosmos_runner.download(MODELS / "cosmos-embed1-336p")
count = lambda m: sum(p.numel() for p in m.parameters())
print(f"InternVideo2_CLIP_S: vision {count(iv2.vision) + count(iv2.align):,}, text {count(iv2.text):,} parameters; Cosmos weights in {cosmos_dir}")

In [ ]:
# 3 · The Cosmos process (transformers 4.51.3): text vectors, GPU time, then both spacings. Resumable
import time
SESSION = time.strftime("%Y-%m-%d %H:%M") + " " + torch.cuda.get_device_name(0)
proc = subprocess.Popen(
    [sys.executable, str(REPO_DIR / "scripts" / "encode_virat_cosmos.py"), "cosmos",
     "--repo", str(REPO_DIR), "--model-dir", str(cosmos_dir), "--videos", str(VIDEOS),
     "--source", str(IV2_ARCHIVE), "--out-root", str(COSMOS_ARCHIVE), "--pins", json.dumps(cosmos_pins),
     "--session", SESSION],
    env={**os.environ, "PYTHONPATH": "/content/cosmos_env"},
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
for line in proc.stdout:
    print(line, end="")
assert proc.wait() == 0, "the Cosmos process failed: see its output above"

In [ ]:
# 4 · GPU time per clip: InternVideo2 here (1 s, float32, as the archive ran) on the same 64 clips Cosmos was timed on
KEYS = archive_runner.evaluation_keys(REPO_DIR)
assert len(KEYS) == 275
cosmos_time = json.loads((COSMOS_ARCHIVE / cosmos_runner.TIMING).read_text())
assert cosmos_time["timed_video"] == KEYS[0]
clips = cosmos_runner.sample_clips(VIDEOS / f"{KEYS[0]}.mp4", IV2_ARCHIVE, KEYS[0], clip_encoder, None, 64)
ms = {"InternVideo2_CLIP_S, 1s, float32": cosmos_runner.time_per_clip(iv2.encode_clips, clips)}
del clips
ms |= {f"Cosmos-Embed1-336p, {s}, bfloat16": v for s, v in cosmos_time["ms_per_clip"].items()}
videos = archive_runner.read_manifest(IV2_ARCHIVE)["videos"]
clips_per_camera_s = sum(v["admitted"] for v in videos.values()) / sum(v["frames_decoded"] / v["fps"] for v in videos.values())
cameras = {name: 1000 / (value * clips_per_camera_s) for name, value in ms.items()}
print(f"{KEYS[0]}, 64 clips, batches of {cosmos_runner.BATCH}; a camera asks for {clips_per_camera_s:.2f} clips per second (VIRAT archive average)")
for name, value in ms.items():
    print(f"  {name:36s} {value:7.1f} ms per clip   about {cameras[name]:5.1f} cameras per {torch.cuda.get_device_name(0)} (clip stream only)")
timing = {"timed_video": KEYS[0], "clips": 64, "batch": cosmos_runner.BATCH, "ms_per_clip": ms,
          "clips_per_camera_s": clips_per_camera_s, "cameras_per_gpu_clip_stream_only": cameras}

In [ ]:
# 5 · Each Cosmos archive, whole, and the labels before any search (OMNI-709: 25.1%, actions 11.7%, scenes 38.5%)
SPACINGS = cosmos_runner.SPACINGS
queries = sm1.read_queries(REPO_DIR / "datasets" / "eval" / "sm1_queries.csv")
dev_queries = sm1.read_queries(REPO_DIR / "datasets" / "eval" / "sm1_dev_queries.csv")
assert len(queries) == 50 and len(dev_queries) == 50
rules = {q["answer_rule"] for q in queries + dev_queries}
loaded = {s: sm1.load_archive(KEYS, COSMOS_ARCHIVE / s, VIRAT, rules, pins=cosmos_pins) for s in SPACINGS}
for s, archive in loaded.items():
    print(f"{s}: {len(archive):,} frames, clip vectors {archive.clip_vectors.shape}")
    assert len(archive) == 35_704, "OMNI-708 archived 35,704 admitted frames"
chance = {q["query_id"]: sm1.chance_at_5(loaded["1s"].correct[q["answer_rule"]]) for q in queries}
average = lambda kind: sum(chance[q["query_id"]] for q in queries if q["kind"] == kind) / 25
print(f"chance: all {sum(chance.values()) / 50:.1%}, actions {average('action'):.1%}, scenes {average('scene'):.1%}")
assert (round(sum(chance.values()) / 50, 3), round(average("action"), 3), round(average("scene"), 3)) == (0.251, 0.117, 0.385)
cosmos_text = cosmos_runner.StoredTexts(COSMOS_ARCHIVE / cosmos_runner.TEXTS)

In [ ]:
# 6 · Tune the weights per spacing on the tuning queries only (D-40's grid and order), never on SM-1's
dev_texts = [q["text"] for q in dev_queries]
dev_frame, dev_clip = siglip.encode_texts(dev_texts), cosmos_text.encode_texts(dev_texts)
tuning = {s: cosmos_runner.tune(dev_queries, loaded[s], dev_frame, dev_clip, classifier, sm1, tune_fusion) for s in SPACINGS}
for s, t in tuning.items():
    print(f"=== {s}")
    print("frame only", t["frame_only"])
    print("clip only ", t["clip_only"])
    print("chosen    ", t["best"])

In [ ]:
# 7 · SM-1 per spacing with its own weights; the better spacing; the rule
def weights_of(best):
    return lambda w_f: (best["scene_w_f"], 1 - best["scene_w_f"]) if w_f >= 0.5 else (best["action_w_f"], 1 - best["action_w_f"])

results = {}
for s in SPACINGS:
    best = tuning[s]["best"]
    results[s] = sm1.measure(queries, loaded[s], siglip, cosmos_text, classifier, weights=weights_of(best), depth=best["depth"])
print(f"{'':12s}{'frame':>8s}{'clip':>8s}{'fused':>8s}{'as B.3':>8s}{'equal':>8s}{'chance':>8s}{'fused 2 s':>11s}")
for s, result in results.items():
    for name, row in [("all", result["overall"]), ("scene", result["by_kind"]["scene"]), ("action", result["by_kind"]["action"])]:
        print(f"{s + ' ' + name:12s}{row['frame']:8.1%}{row['clip']:8.1%}{row['fused']:8.1%}{row['fused_classifier']:8.1%}{row['fused_equal']:8.1%}{row['chance']:8.1%}{row['near']['fused']:11.1%}")
print("InternVideo2 1s (OMNI-713, D-40): fused 76.0% overall; actions fused 60.0%, frame only 60.0%; scenes 92.0%")

# The better spacing, written before the run: higher fused overall, then higher fused on actions, then A.
better = max(SPACINGS, key=lambda s: (results[s]["overall"]["fused"], results[s]["by_kind"]["action"]["fused"], s == "1s"))
r = results[better]
lift = r["by_kind"]["action"]["fused"] > r["by_kind"]["action"]["frame"]
keeps = r["overall"]["fused"] >= 0.76
print(f"\nbetter spacing: {better}; rule 1, action lift over frame only: {lift}; rule 2, fused overall at least 76.0%: {keeps}")
print(f"SWITCH to Cosmos-Embed1, spacing {better}" if lift and keeps else "KEEP InternVideo2")
print()
for s, result in results.items():
    print(f"=== {s}")
    for row in result["queries"]:
        marks = " ".join(f"{m}:{row['modes'][m]['first_correct_rank'] or '-'}" for m in sm1.MODES)
        print(f"{row['query_id']} {row['kind']:6s} {marks}  {row['text']}")

In [ ]:
# 8 · Save to Drive
out_data = {
    "spacings": {
        s: {**results[s], "tuning": {k: v for k, v in tuning[s].items() if k != "grid"} | {"grid_top_10": tuning[s]["grid"][:10]},
            "encoding": cosmos_runner.timing(COSMOS_ARCHIVE / s), "span_s": SPACINGS[s]}
        for s in SPACINGS
    },
    "gpu_time": timing,
    "better": better,
    "rule": {"action_lift": bool(lift), "fused_at_least_76": bool(keeps), "switch": bool(lift and keeps)},
    "conditions": {
        "commit": COMMIT, "gpu": torch.cuda.get_device_name(0), "torch": torch.__version__, "pins": cosmos_pins,
        "cosmos_dtype": "bfloat16", "cosmos_transformers": cosmos_time["transformers"],
        "chosen_on_sha256": sm1.file_sha256(REPO_DIR / "datasets" / "eval" / "sm1_dev_queries.csv"),
        "queries_sha256": sm1.file_sha256(REPO_DIR / "datasets" / "eval" / "sm1_queries.csv"),
        "classifier_sha256": sm1.file_sha256(CLASSIFIER_FILE), "k": sm1.K, "search": "exact, every vector compared",
    },
}
out = DRIVE / "results" / "sm1_result_cosmos.json"
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(out_data, indent=1))
print(out, out_data["better"], out_data["rule"])